# 02. Prompt Tuning with the Same Fast Model

Goal: improve quality by prompt engineering before changing model tier.


## Step 1: Install Dependencies


In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r requirements.txt


## Step 2: Load Shared Utilities


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/ESMT_Workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report

print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Configure API Client


In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ['WORKSHOP_EMAIL']

In [ ]:
# API access is controlled by email at the proxy server.
# No API key is required in this workshop implementation.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')

MODEL_NAME = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = 40
MAX_TOKENS = 512
MAX_WORKERS = 8


## Step 4: Define Reusable Experiment Function


In [ ]:
from esmt_workshop.student_api import process_batch_addresses


def run_experiment(prompt_template: str, *, temperature: float = 0.0, top_p: float = 1.0, top_k: int = 40):
    # Load development labels for prompt iteration.
    dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')

    # Use the same model while changing prompt and sampling controls.
    pred_df = process_batch_addresses(
        dev_df,
        email=STUDENT_EMAIL,
        stage='prompt_tuned',
        model=MODEL_NAME,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        max_tokens=MAX_TOKENS,
        custom_prompt_template=prompt_template,
        max_workers=MAX_WORKERS,
    )

    report = evaluate_predictions(pred_df, dev_df, email=STUDENT_EMAIL)
    return pred_df, report


## Step 5: Run the Starter Prompt


In [ ]:
# Starter prompt editable directly in code.
STARTER_PROMPT_TEMPLATE = '''You are an address parser for AML compliance.

Input address:
{address}

Return strict JSON only using this schema:
{schema}

Rules:
1. Town Name is only city/town/locality.
2. Postal Code includes only postal token(s).
3. Remaining Address includes everything else.
4. Country Code is ISO alpha-2 uppercase.
5. Use empty strings when uncertain.
'''

pred_v1, report_v1 = run_experiment(STARTER_PROMPT_TEMPLATE, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K)

print('v1 summary:', report_v1['summary'])
report_v1['field_metrics']


## Step 6: Run Your Team Prompt


In [ ]:
# Team prompt editable directly in this notebook cell.
STUDENT_PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Extract Town Name, Postal Code, Remaining Address, Country Code (2 characters).
- Country Code must be ISO alpha-2 uppercase.
- Do not invent values.
- Use empty strings if unknown.
'''

pred_student, report_student = run_experiment(
    STUDENT_PROMPT_TEMPLATE,
    temperature=0.0,
    top_p=0.95,
    top_k=50,
)
print('student summary:', report_student['summary'])
report_student['field_metrics']


## Step 7: Save Prompt-Tuning Artifacts


In [ ]:
# Save prompt-tuning artifacts for both prompt versions.
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

pred_v1.to_csv(out_dir / '02_prompt_tuned_v1_predictions.csv', index=False)
save_evaluation_report(report_v1, out_dir / 'report_02_prompt_tuned_v1')

pred_student.to_csv(out_dir / '02_prompt_tuned_student_predictions.csv', index=False)
save_evaluation_report(report_student, out_dir / 'report_02_prompt_tuned_student')

print('Saved prompt-tuning artifacts.')


## Assignment

1. Edit `STUDENT_PROMPT_TEMPLATE` in this notebook and rerun.
2. Compare `micro_accuracy` and mismatch profiles.
3. Move to `03_advanced_model.ipynb`.
